# ESC-Daten herunterladen

Lädt die Eurovision-Daten aus den GitHub Releases von [Spijkervet/eurovision-dataset](https://github.com/Spijkervet/eurovision-dataset) nach `data/`.
Kein Kaggle-Login, keine API-Keys nötig.

Hinweis: Der Release ist ein eingefrorener Stand (neuester Tag: `2023`). Neuere Jahrgänge gibt es nur, wenn man den Scraper im Repo selbst laufen lässt.

Kaggle-Alternative: https://www.kaggle.com/datasets/diamondsnake/eurovision-song-contest-data

In [ ]:
import json
import urllib.request
from pathlib import Path

REPO = "Spijkervet/eurovision-dataset"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## Verfügbare Releases anzeigen

In [ ]:
with urllib.request.urlopen(f"https://api.github.com/repos/{REPO}/releases") as r:
    releases = json.load(r)

for rel in releases:
    print(rel["tag_name"], [a["name"] for a in rel["assets"]])

## Dateien herunterladen

`TAG = None` nimmt automatisch den neuesten Release; alternativ z. B. `TAG = "2023"` fest setzen.

In [ ]:
TAG = None
FILES = ["contestants.csv", "votes.csv", "betting_offices.csv"]

release = releases[0] if TAG is None else next(r for r in releases if r["tag_name"] == TAG)
print("Release:", release["tag_name"])

assets = {a["name"]: a["browser_download_url"] for a in release["assets"]}
for name in FILES:
    target = DATA_DIR / name
    urllib.request.urlretrieve(assets[name], target)
    print(f"{name:22s} {target.stat().st_size / 1e6:5.2f} MB")

## Kurzer Check

In [ ]:
import pandas as pd

for name in FILES:
    df = pd.read_csv(DATA_DIR / name)
    print(f"{name:22s} {df.shape}  Jahre: {df['year'].min()}–{df['year'].max()}")